# `01_data_generation.ipynb` — MIND data generation for `meta-llama/Llama-2-7b-hf`

**Model:** `meta-llama/Llama-2-7b-hf`  **Samples / class:** `200`  **Output:** `kaggle_llama2_7b_dataset_full.json`

This notebook generates the MIND-style hallucination-detection training dataset.
**It is the slow common stage** — after it finishes, you can run the two analysis
notebooks IN PARALLEL on two different accounts.

## Workflow

```
                  [account 1 / first session]
                  Run 01_data_generation.ipynb (~25-50 min)
                            │
                            ▼
                  Download <tag>_dataset_full.json
                            │
                ┌───────────┴───────────┐
                ▼                       ▼
        [account A / session A]   [account B / session B]
        Upload dataset_full       Upload dataset_full
        Run 02_all_variants       Run 03_baselines_sota
        (~45-60 min)              (~85-90 min)
                │                       │
                └───────────┬───────────┘
                            ▼
              Two result JSONs, paste back to assistant.
```

## What gets saved

* `kaggle_llama2_7b_dataset_full.json` — every MIND record (text + canonical embedding + D_mean + V_last + H_mean per record). This is the file you download and re-upload to the second account.

That's it — no MLP training, no multi-task evaluation. Those happen in 02 and 03.

## Why split?

The LLM data-gen step (running `model.generate()` per Wikipedia article with the MIND Algorithm-1 entity-substitution loop) is the slowest stage by ~50%. Doing it once and feeding the dataset into two parallel analysis runs saves you ~30-45 min wall-clock per model.


In [ ]:
# =============================================================================
# BLOCK 0: PIP INSTALLS  +  (conditional) HuggingFace login for gated models
# =============================================================================
!pip install -q -U "transformers>=4.45" "tokenizers>=0.19" "accelerate>=0.30"
!pip install -q -U datasets spacy nltk scikit-learn tqdm sentence-transformers
!python -m spacy download en_core_web_sm

# -------- HuggingFace login (REQUIRED for gated models e.g. meta-llama/*) ----
# If meta-llama/Llama-2-7b-hf is gated you MUST:
#   1. Accept the model's license on huggingface.co (one-time per account).
#   2. Generate a User Access Token at huggingface.co/settings/tokens.
#   3. Paste the token when the cell below prompts.
# For non-gated models (gpt2, Qwen, etc.) the cell is a no-op.
_GATED_PREFIXES = ("meta-llama/", "google/gemma", "mistralai/")
if any("meta-llama/Llama-2-7b-hf".startswith(p) for p in _GATED_PREFIXES):
    from huggingface_hub import login
    login()   # opens an input box. Paste your HF token (starts with hf_...).
else:
    print("meta-llama/Llama-2-7b-hf is not gated -- skipping HF login.")


In [ ]:
# =============================================================================
# BLOCK 1 (STAGE 1): SETUP — imports, config, diagnostics dict, lazy LLM loader
# =============================================================================
import os, sys, gc, json, random, math, time, datetime, platform, traceback
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

MODEL_NAME       = "meta-llama/Llama-2-7b-hf"
MODEL_TAG        = "kaggle_llama2_7b"
TARGET_SAMPLES   = 200       # per class
TOPK_FIRST_TOKEN = 4
WINDOWS          = 16
SEED             = 42
DTYPE            = torch.bfloat16 if torch.cuda.is_available() else torch.float32
LOGIT_LENS_LAYER = 16       # for F5 (DoLa-style)
MAX_GEN_NEW      = 48
EPOCHS           = 10
BATCH            = 32
LR               = 5e-4
WD               = 1e-5
TEST_FRAC        = 0.20

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

PATH_DATASET_FULL     = f"{MODEL_TAG}_dataset_full.json"
PATH_DATASET_FEATURES = f"{MODEL_TAG}_dataset_with_features.json"
PATH_RESULTS          = f"{MODEL_TAG}_all_variants_results.json"
PATH_CHECKPOINT       = f"{MODEL_TAG}_datagen_checkpoint.json"

DIAG = {
    "schema_version": "all-variants-2.0",
    "model_tag":  MODEL_TAG,
    "model_name": MODEL_NAME,
    "config": {
        "target_samples_per_class": TARGET_SAMPLES,
        "topk_first_token": TOPK_FIRST_TOKEN, "windows": WINDOWS, "seed": SEED,
        "dtype": str(DTYPE), "logit_lens_layer": LOGIT_LENS_LAYER,
        "max_gen_new_tokens": MAX_GEN_NEW,
        "epochs": EPOCHS, "batch_size": BATCH,
        "lr": LR, "weight_decay": WD, "test_frac": TEST_FRAC,
    },
    "env": {
        "python":   sys.version.split()[0],
        "platform": platform.platform(),
        "torch":    torch.__version__,
        "cuda_available": bool(torch.cuda.is_available()),
        "cuda_device":   (torch.cuda.get_device_name(0) if torch.cuda.is_available() else None),
        "free_vram_gb":  (round(torch.cuda.mem_get_info()[0]/1e9, 2) if torch.cuda.is_available() else None),
    },
    "library_versions": {},
    "stage_timings_sec": {},
    "downstream_datasets": {},
    "errors": [],
}
for _mod in ["transformers", "datasets", "accelerate", "sklearn", "spacy", "nltk", "numpy"]:
    try:
        m = __import__(_mod)
        DIAG["library_versions"][_mod] = getattr(m, "__version__", "n/a")
    except Exception as e:
        DIAG["library_versions"][_mod] = f"IMPORT_FAILED: {e}"

_MODEL = {"tokenizer": None, "model": None, "nlp": None}

def need_llm():
    """Lazy-load the LLM + spacy. Idempotent."""
    if _MODEL["model"] is not None:
        return _MODEL
    from transformers import AutoTokenizer, AutoModelForCausalLM
    import spacy, nltk
    nltk.download("punkt", quiet=True); nltk.download("punkt_tab", quiet=True)
    print(f"Loading {MODEL_NAME} (dtype={DTYPE}) ...")
    _t0 = time.perf_counter()
    tok = AutoTokenizer.from_pretrained(MODEL_NAME)
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    # IMPORTANT: attn_implementation='eager' is required because Stage 3
    # extracts F1/F2/F6 from output_attentions=True, which SDPA does NOT
    # support. Without this, Qwen2.5 / Llama-2 / Falcon return None for
    # attentions and every F-feature silently falls back to 0.
    load_kwargs = dict(torch_dtype=DTYPE,
                       device_map="auto" if torch.cuda.is_available() else None,
                       attn_implementation="eager")
    try:
        mdl = AutoModelForCausalLM.from_pretrained(MODEL_NAME, **load_kwargs)
    except (TypeError, ValueError) as _e:
        # Some custom models (e.g. older Falcon w/ trust_remote_code) reject
        # the kwarg. Retry without it; F1/F2/F6 may then be 0/NaN but the
        # other features and the rest of the pipeline still complete.
        print(f"[warn] retrying model load without attn_implementation: {_e}")
        load_kwargs.pop("attn_implementation", None)
        mdl = AutoModelForCausalLM.from_pretrained(MODEL_NAME, **load_kwargs)
    mdl.eval()
    for p in mdl.parameters(): p.requires_grad = False
    nlp = spacy.load("en_core_web_sm")
    DIAG["stage_timings_sec"]["model_load"] = round(time.perf_counter() - _t0, 2)
    DIAG["model_info"] = {
        "hidden_dim":    int(mdl.config.hidden_size),
        "n_layers":      int(mdl.config.num_hidden_layers),
        "n_heads":       int(getattr(mdl.config, "num_attention_heads", -1)),
        "vocab_size":    int(mdl.config.vocab_size),
        "max_pos_embed": int(getattr(mdl.config, "max_position_embeddings", -1)),
        "param_count_M": round(sum(p.numel() for p in mdl.parameters()) / 1e6, 2),
        "device":        str(next(mdl.parameters()).device),
    }
    print(f"✓ model loaded in {DIAG['stage_timings_sec']['model_load']}s   "
          f"hidden_dim={DIAG['model_info']['hidden_dim']}   "
          f"layers={DIAG['model_info']['n_layers']}")
    _MODEL["tokenizer"] = tok; _MODEL["model"] = mdl; _MODEL["nlp"] = nlp
    return _MODEL


print("=" * 80); print(f"STAGE 1 — SETUP — {MODEL_TAG}"); print("=" * 80)
print(f"Resume points:")
print(f"  Stage 2 -> {PATH_DATASET_FULL}      {'EXISTS — SKIP' if os.path.exists(PATH_DATASET_FULL) else 'will run'}")
print(f"  Stage 3 -> {PATH_DATASET_FEATURES}  {'EXISTS — SKIP' if os.path.exists(PATH_DATASET_FEATURES) else 'will run'}")
print(f"  Stage 7 -> {PATH_RESULTS}           {'EXISTS — overwrite' if os.path.exists(PATH_RESULTS) else 'will run'}")


In [ ]:
# =============================================================================
# BLOCK 1.5 (STAGE 1.5): ALWAYS-DEFINED FEATURE-EXTRACTION + ENTITY HELPERS
# Defined unconditionally so Stage 6 (multi-task eval) can call them even when
# Stages 2 and 3 are skipped because their output files already exist.
# =============================================================================

EXTRA_FEATURE_KEYS = [
    "F1_lookback_ratio", "F2_attention_sink", "F3_eigenscore_lite",
    "F4_icr_score", "F5_logit_lens_jsd", "F6_head_entropy",
    "F7_max_margin", "F8_token_rank", "F10_intra_dispersion",
]

# ---------------- entity helpers (used by Stage 2) ----------------
def delete_substrings(lst):
    subs = []; lst = list(set(lst))
    for s in lst:
        if any(s in o for o in lst if o != s): subs.append(s)
    for s in subs: lst.remove(s)
    return lst

def find_boundaries(text, words):
    out = []
    for word in words:
        ntext = text
        while True:
            start = ntext.find(word)
            if start == -1: break
            end = start + len(word) - 1
            while start > 0 and ntext[start-1] != " ": start -= 1
            while end < len(ntext)-1 and ntext[end+1] != " ": end += 1
            out.append("".join(ntext[i] for i in range(start, end+1)))
            ntext = ntext[end+1:]
    return out

def get_entities(text):
    LL = need_llm(); nlp = LL["nlp"]
    ents = list({str(e) for e in nlp(text).ents})
    ents = find_boundaries(text, ents)
    ents = delete_substrings(ents)
    out = []
    for i in range(len(text)):
        for e in ents:
            if text[i:].startswith(e): out.append((e, i))
    return out

def find_first_and_next_token(text, e, idx, input_id, prompt=""):
    LL = need_llm(); tokenizer = LL["tokenizer"]
    new_text = f"{text[:idx].strip()} {text[idx:].replace(e, e+' @', 1).strip()}"
    new_input_id = tokenizer(prompt + new_text.strip(), return_tensors="pt")["input_ids"].tolist()[0]
    for i in range(len(input_id[0])):
        if input_id[0][i] != new_input_id[i]: return []
    first_token = new_input_id[len(input_id[0])]
    at_cands = tokenizer("@", add_special_tokens=False)["input_ids"]
    at_cands += tokenizer(" @", add_special_tokens=False)["input_ids"]
    at_pos = None
    for at_tok in at_cands:
        try:
            at_pos = new_input_id.index(at_tok, len(input_id[0])); break
        except ValueError: continue
    if at_pos is None or at_pos >= len(new_input_id) - 1: return []
    next_token = new_input_id[at_pos + 1]
    entity_len = at_pos - len(input_id[0])
    last_id = new_input_id[at_pos + 1:]
    return [first_token, next_token, entity_len, last_id]


# ---------------- MIND 4 features (canonical + D_mean + V_last + H_last) ----------------
@torch.no_grad()
def extract_mind(text: str):
    LL = need_llm()
    tokenizer, model = LL["tokenizer"], LL["model"]
    enc = tokenizer(text.strip(), return_tensors="pt", truncation=True,
                    max_length=getattr(model.config, "max_position_embeddings", 4096)).to(model.device)
    out = model(**enc, output_hidden_states=True, use_cache=False)
    hs = out.hidden_states
    last_per_layer = torch.stack([h[0, -1, :].float() for h in hs[1:]], dim=0)
    canonical = last_per_layer[-1].cpu().tolist()
    if last_per_layer.shape[0] < 2:
        D_mean = 0.0
    else:
        cos = F.cosine_similarity(last_per_layer[:-1], last_per_layer[1:], dim=1)
        D_mean = float((1.0 - cos).mean().item())
    mean_h = last_per_layer.mean(dim=0)
    V_last = float(((last_per_layer - mean_h) ** 2).sum(dim=1).mean().item())
    last_logits = out.logits[0, -1, :].float()
    p_last = torch.softmax(last_logits, dim=-1)
    H_last = float(-(p_last * p_last.clamp_min(1e-12).log()).sum().item())
    return canonical, D_mean, V_last, H_last


def per_step_entropy(scores):
    if not scores: return 0.0
    Hs = []
    for s in scores:
        p = torch.softmax(s[0].float(), dim=-1)
        Hs.append(float(-(p * p.clamp_min(1e-12).log()).sum().item()))
    return float(sum(Hs) / len(Hs))


# ---------------- 9 F features (F1, F2, F3, F4, F5, F6, F7, F8, F10) ----------------
def _jsd(p, q, eps=1e-12):
    p = p / p.sum(); q = q / q.sum()
    m = 0.5 * (p + q)
    def _kl(a, b):
        return float((a * (a.clamp_min(eps).log() - b.clamp_min(eps).log())).sum().item())
    return 0.5 * _kl(p, m) + 0.5 * _kl(q, m)

_EXTRAS_CACHE = {"sink_ids": None}

def _ensure_extras_cache():
    if _EXTRAS_CACHE["sink_ids"] is not None: return
    LL = need_llm(); tokenizer = LL["tokenizer"]
    sink_strs = ["<|endoftext|>", "<s>", "</s>", "<|im_start|>", "<|im_end|>",
                  ".", ",", "!", "?", ";", ":", " .", " ,", " !", " ?", " ;", " :"]
    sink_ids = set()
    for s in sink_strs:
        try:
            for tid in tokenizer(s, add_special_tokens=False)["input_ids"]:
                sink_ids.add(int(tid))
        except Exception: continue
    for tid in [tokenizer.bos_token_id, tokenizer.eos_token_id, tokenizer.pad_token_id]:
        if tid is not None: sink_ids.add(int(tid))
    _EXTRAS_CACHE["sink_ids"] = sink_ids


@torch.no_grad()
def extract_extras(text: str):
    LL = need_llm()
    tokenizer, model = LL["tokenizer"], LL["model"]
    _ensure_extras_cache()
    SINK_IDS = _EXTRAS_CACHE["sink_ids"]
    idx = text.find(". ")
    prompt_text = text[: (idx + 2 if idx != -1 else len(text)//2)]
    enc = tokenizer(text.strip(), return_tensors="pt", truncation=True,
                    max_length=getattr(model.config, "max_position_embeddings", 4096)).to(model.device)
    input_ids = enc.input_ids; T = input_ids.shape[1]
    p_enc = tokenizer(prompt_text.strip(), return_tensors="pt", truncation=True,
                       max_length=getattr(model.config, "max_position_embeddings", 4096)).input_ids
    prompt_len = max(1, min(p_enc.shape[1], T - 1))
    out = model(**enc, output_hidden_states=True, output_attentions=True, use_cache=False)
    hs = out.hidden_states; attn = out.attentions; logits = out.logits

    last_layer_attn = attn[-1][0]
    last_pos = last_layer_attn[:, -1, :]
    mass_p = last_pos[:, :prompt_len].sum(dim=1).float()
    mass_g = last_pos[:, prompt_len:].sum(dim=1).float()
    F1 = float((mass_p / (mass_p + mass_g).clamp_min(1e-8)).mean().item())

    sink_mask = torch.zeros(T, dtype=torch.bool, device=input_ids.device)
    for i, tid in enumerate(input_ids[0].tolist()):
        if int(tid) in SINK_IDS: sink_mask[i] = True
    if sink_mask.any():
        sm = []
        for la in attn:
            a = la[0, :, -1, :].float()
            sm.append(float((a[:, sink_mask].sum(dim=1) / a.sum(dim=1).clamp_min(1e-8)).mean().item()))
        F2 = float(sum(sm) / len(sm))
    else:
        F2 = 0.0

    H_last = hs[-1][0].float()
    Hc = H_last - H_last.mean(dim=0, keepdim=True)
    if T >= 2:
        cov_t = (Hc @ Hc.t()) / max(T - 1, 1)
        cov_t = cov_t + 1e-3 * torch.eye(T, device=cov_t.device, dtype=cov_t.dtype)
        try:
            sgn, logabsdet = torch.slogdet(cov_t)
            F3 = float(logabsdet.item() if sgn > 0 else float("nan"))
        except Exception: F3 = float("nan")
    else: F3 = float("nan")

    ratios = []
    for l in range(1, len(hs)):
        h_prev = hs[l-1][0, -1, :].float(); h_curr = hs[l][0, -1, :].float()
        d = float(h_curr.norm().item())
        if d <= 1e-8: continue
        ratios.append(float((h_curr - h_prev).norm().item()) / d)
    F4 = float(sum(ratios) / max(len(ratios), 1))

    try:
        lm_w = model.get_output_embeddings().weight.float()
        ref = min(LOGIT_LENS_LAYER, len(hs) - 1)
        ph = hs[ref][0, -1, :].float()
        pe = torch.softmax(ph @ lm_w.t(), dim=-1)
        pf = torch.softmax(logits[0, -1, :].float(), dim=-1)
        F5 = _jsd(pe, pf)
    except Exception: F5 = float("nan")

    he = []
    for la in attn:
        a = la[0, :, -1, :].float().clamp_min(1e-12)
        a = a / a.sum(dim=1, keepdim=True).clamp_min(1e-12)
        he.append(float((-(a * a.log()).sum(dim=1)).mean().item()))
    F6 = float(sum(he) / max(len(he), 1))

    if T >= 2:
        probs = torch.softmax(logits[0, :-1, :].float(), dim=-1)
        top2 = probs.topk(2, dim=-1).values
        F7 = float((top2[:, 0] - top2[:, 1]).mean().item())
    else: F7 = 0.0

    if T >= 2:
        probs = torch.softmax(logits[0, :-1, :].float(), dim=-1)
        tgt = input_ids[0, 1:]; ranks = []
        si = probs.argsort(dim=-1, descending=True)
        for i in range(T - 1):
            pos = (si[i] == tgt[i]).nonzero(as_tuple=True)
            if len(pos[0]) > 0: ranks.append(float(pos[0][0].item() + 1))
        F8 = float(sum(ranks) / max(len(ranks), 1))
    else: F8 = 0.0

    if T >= 2:
        Hf = hs[-1][0].float()
        c = Hf.mean(dim=0, keepdim=True)
        F10 = float((Hf - c).pow(2).sum(dim=1).sqrt().mean().item())
    else: F10 = 0.0

    return {"F1_lookback_ratio": F1, "F2_attention_sink": F2,
            "F3_eigenscore_lite": F3, "F4_icr_score": F4,
            "F5_logit_lens_jsd": F5, "F6_head_entropy": F6,
            "F7_max_margin": F7, "F8_token_rank": F8,
            "F10_intra_dispersion": F10}


print("✓ feature helpers defined (extract_mind, extract_extras, get_entities, ...)")


In [ ]:
# =============================================================================
# BLOCK 2 (STAGE 2): MIND DATA GENERATION  (skip if dataset_full.json exists)
# =============================================================================
if os.path.exists(PATH_DATASET_FULL):
    print(f"[SKIP STAGE 2] {PATH_DATASET_FULL} exists — loading.")
    _t0 = time.perf_counter()
    with open(PATH_DATASET_FULL, "r") as _f:
        records_full = json.load(_f)
    DIAG["stage_timings_sec"]["data_gen_skipped_load"] = round(time.perf_counter() - _t0, 2)
    n_h1 = sum(1 for r in records_full if r["label"] == 1)
    n_h0 = sum(1 for r in records_full if r["label"] == 0)
    DIAG["data_gen"] = {"loaded_from_disk": True, "n_hallucinated": n_h1,
                         "n_non_hallucinated": n_h0, "n_total": len(records_full)}
    print(f"  {len(records_full)} records  (y=1:{n_h1}  y=0:{n_h0})")
else:
    print(f"Generating MIND dataset -> {PATH_DATASET_FULL}")
    LL = need_llm()
    tokenizer, model = LL["tokenizer"], LL["model"]
    from datasets import load_dataset
    from nltk.tokenize import sent_tokenize
    from tqdm.auto import tqdm

    SKIP = {"overflow_prefix": 0, "no_token_match": 0, "overflow_suffix": 0,
             "model_knew": 0, "no_match_in_topk": 0, "empty_entity": 0,
             "duplicate_entity": 0, "exception": 0}

    def generate_sample(text, entity, idx, title):
        MAX_POS = getattr(model.config, "max_position_embeddings", 4096)
        HEADROOM = WINDOWS + 64
        enc = tokenizer(text[:idx].strip(), return_tensors="pt").to(model.device)
        input_ids = enc["input_ids"]; attn = enc["attention_mask"]
        if input_ids.shape[1] + HEADROOM >= MAX_POS:
            SKIP["overflow_prefix"] += 1; return None
        toks = find_first_and_next_token(text, entity, idx, input_ids.tolist())
        if not toks: SKIP["no_token_match"] += 1; return None
        first_, next_, entity_len, last_id = toks
        if input_ids.shape[1] + entity_len + len(last_id) + HEADROOM >= MAX_POS:
            SKIP["overflow_suffix"] += 1; return None
        gen = model.generate(input_ids, attention_mask=attn,
                             max_new_tokens=entity_len + WINDOWS,
                             return_dict_in_generate=True, output_scores=True,
                             do_sample=False, pad_token_id=tokenizer.eos_token_id)
        if first_ in torch.topk(gen.scores[0], k=TOPK_FIRST_TOKEN).indices[0].tolist():
            SKIP["model_knew"] += 1; return None
        found_step = None
        for step, sc in enumerate(gen.scores):
            if next_ in torch.topk(sc, k=TOPK_FIRST_TOKEN).indices[0].tolist():
                found_step = step; break
        if found_step is None: SKIP["no_match_in_topk"] += 1; return None
        H_mean_hall = per_step_entropy(gen.scores[: found_step + 1])
        new_seq = gen.sequences[0, : input_ids.shape[1] + found_step].tolist()
        new_entity_ids = new_seq[input_ids.shape[1]:]
        hallucinated_ids = input_ids.tolist()[0] + new_entity_ids + last_id
        hallucinated_text = tokenizer.decode(hallucinated_ids, skip_special_tokens=True)
        new_entity = tokenizer.decode(new_entity_ids, skip_special_tokens=True).strip().lower()
        if not new_entity: SKIP["empty_entity"] += 1; return None
        if entity.lower() in new_entity or new_entity in text.lower():
            SKIP["duplicate_entity"] += 1; return None
        canon_h, D_h, V_h, _    = extract_mind(hallucinated_text)
        canon_o, D_o, V_o, H_o  = extract_mind(text)
        return {"text_hall": hallucinated_text, "entity_hall": new_entity,
                "canon_h": canon_h, "D_h": D_h, "V_h": V_h, "H_h": H_mean_hall,
                "text_orig": text, "entity_orig": entity,
                "canon_o": canon_o, "D_o": D_o, "V_o": V_o, "H_o": H_o,
                "title": title}

    wiki = load_dataset("wikimedia/wikipedia", "20231101.en", split="train", streaming=True)
    dataset_hall, dataset_non_hall = [], []
    processed = 0
    pbar = tqdm(total=TARGET_SAMPLES * 2, desc="samples")
    _t0 = time.perf_counter()
    for row in wiki:
        if len(dataset_hall) >= TARGET_SAMPLES and len(dataset_non_hall) >= TARGET_SAMPLES:
            break
        processed += 1
        try:
            sents = sent_tokenize(row["text"])
            if len(sents) < 2: continue
            text = " ".join(sents[:2]); title = row.get("title", "")
            ents = get_entities(text)
            if not ents: continue
            seen, ents_filt = set(), []
            for e, i in ents:
                if i == 0 or e.lower() in title.lower(): continue
                if i not in seen: seen.add(i); ents_filt.append((e, i))
            if not ents_filt: continue
            entity, char_idx = random.choice(ents_filt)
            result = generate_sample(text, entity, char_idx, title)
            if result is None: continue
            if len(dataset_hall) < TARGET_SAMPLES:
                dataset_hall.append({"label": 1, "text": result["text_hall"],
                                      "entity": result["entity_hall"],
                                      "embedding": result["canon_h"],
                                      "D_mean": result["D_h"], "V_last": result["V_h"],
                                      "H_mean": result["H_h"], "title": result["title"]})
                pbar.update(1)
            if len(dataset_non_hall) < TARGET_SAMPLES:
                dataset_non_hall.append({"label": 0, "text": result["text_orig"],
                                          "entity": result["entity_orig"],
                                          "embedding": result["canon_o"],
                                          "D_mean": result["D_o"], "V_last": result["V_o"],
                                          "H_mean": result["H_o"], "title": result["title"]})
                pbar.update(1)
            pbar.set_postfix(H1=len(dataset_hall), H0=len(dataset_non_hall), proc=processed)
            if (len(dataset_hall) + len(dataset_non_hall)) % 200 == 0:
                with open(PATH_CHECKPOINT, "w") as _f:
                    json.dump(dataset_hall + dataset_non_hall, _f)
        except Exception:
            SKIP["exception"] += 1; continue
        if processed % 100 == 0 and torch.cuda.is_available():
            torch.cuda.empty_cache()
    pbar.close()
    DIAG["stage_timings_sec"]["data_gen"] = round(time.perf_counter() - _t0, 2)
    records_full = dataset_hall + dataset_non_hall
    with open(PATH_DATASET_FULL, "w") as _f:
        json.dump(records_full, _f)
    DIAG["data_gen"] = {
        "loaded_from_disk": False, "wiki_articles_processed": processed,
        "n_hallucinated": len(dataset_hall), "n_non_hallucinated": len(dataset_non_hall),
        "n_total": len(records_full), "skip_counters": dict(SKIP),
        "example_hall":     (dataset_hall[0]["text"][:280]     if dataset_hall else None),
        "example_non_hall": (dataset_non_hall[0]["text"][:280] if dataset_non_hall else None),
    }
    print(f"\n✓ wrote {PATH_DATASET_FULL}  ({os.path.getsize(PATH_DATASET_FULL)/1e6:.2f} MB)")
    print(f"  data_gen time: {DIAG['stage_timings_sec']['data_gen']} s   skip: {SKIP}")


In [ ]:
# =============================================================================
# DONE — verify dataset_full.json was written, then proceed to 02 and 03 on
# two separate accounts in parallel.
# =============================================================================
import os
if os.path.exists(PATH_DATASET_FULL):
    sz_mb = os.path.getsize(PATH_DATASET_FULL) / 1e6
    print(f"OK  {PATH_DATASET_FULL}  ({sz_mb:.2f} MB)")
    print(f"   {DIAG['data_gen'].get('n_total', '?')} records  "
          f"(y=1: {DIAG['data_gen'].get('n_hallucinated', '?')}, "
          f"y=0: {DIAG['data_gen'].get('n_non_hallucinated', '?')})")
    print()
    print("NEXT STEPS")
    print("-" * 60)
    print(f"1. Download {PATH_DATASET_FULL} from this session.")
    print(f"2. Upload the file to TWO accounts (Colab or Kaggle).")
    print(f"3. Account A: upload + run 02_all_variants.ipynb   (~45-60 min)")
    print(f"4. Account B (PARALLEL): upload + run 03_baselines_sota.ipynb  (~85-90 min)")
    print()
    print(f"After both finish, paste the two result JSONs back to the assistant.")
else:
    print("ERROR: dataset_full.json was NOT written. Check Stage 2 output for errors.")
